# 4.2 Smolagents use 

### SmolAgents Overview

**Overview**: Hugging Face's open-source Agent framework focused on **code-first task execution**, supporting rapid creation of highly capable, secure agents. Its core philosophy is "code as action" -- using generated code to complete tasks, suitable for tool calling and multi-step reasoning scenarios.

---

#### **Key Features**
1. **Code-First Execution**
- Defines Agent actions through Python code rather than JSON/text instructions, reducing steps by ~30%. For example, calling `CodeAgent` to generate and execute code for calculations or search tasks.
- Supports local and sandboxed (E2B) execution for security.

2. **Rich Multimodal Tool Ecosystem**
- 50+ built-in tools (DuckDuckGo search, Google Image API), supports custom tools via `@tool` decorator for quick registration.
- Integrates with Hugging Face Hub for sharing and reusing tools, e.g., calling community-uploaded tools.

3. **Flexible Model Support**
- Supports Transformers, Ollama for local deployment, and OpenAI, Anthropic APIs via LiteLLM for unified interface calling.
- Example: Quickly switch between `HfApiModel` (cloud) and `TransformersModel` (local) Agents.

4. **Safety and Secure Execution**
- Based on ReAct pattern with multi-step reasoning loops, supports step limits (default 6) and inter-step validation for controlled task execution.
- Sandboxed environment operations ensure safe deployment.

---

#### **Use Cases**
1. **Search and Data Analysis**
- Quickly build retrieval Agents (news search, knowledge queries), generate reports through API calls.
- Supports SQL generation, PDF parsing and other tools for data analysis workflows.

2. **Code Generation and Debugging**
- Generate code (including tests), debug, and execute Python via `CodeAgent` to verify results.

3. **Workflow Automation**
- Multi-agent collaboration, e.g., a planning Agent and an execution Agent working together through tool calling.

---

#### **Advantages**
1. **High Development Efficiency**
- Build an Agent in 3 lines of code (with search tools), low barrier to entry.
- Supports quick debugging, e.g., monitoring execution via `step_callbacks`.

2. **Community Ecosystem**
- Rich documentation and examples (search, code generation), easy for newcomers to get started.

3. **Security and Portability**
- Simple `pip install smolagents`, no heavy dependencies, supports local and cloud deployment.
- Control execution permissions through `additional_authorized_imports` for security.

---

### Summary
SmolAgents excels with its lightweight design and rich tool ecosystem for Agent development. Compared to full-stack frameworks like Phidata, SmolAgents focuses on **code-first task execution**, ideal for scenarios requiring fast response and tool integration. Visit its [GitHub](https://github.com/huggingface/smolagents) for the latest updates.

In [ ]:
#!pip install smolagents
#!pip install smolagents[litellm]

## Code Agents

Code Agents are the primary agent type in smolagents. They generate Python code for tool calling and operation execution, achieving high efficiency where each operation represents a meaningful action.

This approach requires fewer operations, is more composable, and enables the use of existing functions. smolagents provides everything needed to build code agents in under 1,000 lines of code.

<img src="img/code_vs_json_actions.png" width='720px' />

In drone multi-step agent scenarios, the LLM executes operations and calls external tools. Traditional methods use JSON format to describe tool names and parameter strings, which must be parsed before the tool can be executed.

Code-based tool calling is what LLMs are naturally trained to do. This is the core idea behind smolagents.

Using code over JSON actions provides key advantages:

- **Composability**: Easily chain and combine operations
- **Object Management**: Handle complex objects like images -- essential for drone Agents
- **Generality**: Can express any computation
- **LLM Affinity**: Code generation is well-represented in LLM training data

### Hugging Face SmolAgents Resources

Agent Course: https://huggingface.co/learn/agents-course/en/unit2/smolagents/introduction

SmolAgents Documentation: https://huggingface.co/docs/smolagents/v1.11.0/index

GitHub: https://github.com/huggingface/smolagents

In [ ]:
# LLM model, supports local deployment, OpenAI-compatible APIs, etc.

from smolagents import CodeAgent, LiteLLMModel

model = LiteLLMModel(
    model_id="openai/moonshotai/Kimi-K2.6",  # This model is a bit weak for agentic behaviours though
    api_base="https://api.intelligence.io.solutions/api/v1", # replace with 127.0.0.1:11434 or remote open-ai compatible server if necessary
    api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxx", # Replace with your own API key
)

LiteLLMModel tutorial: https://docs.litellm.ai/docs/providers/volcano

In [ ]:
from smolagents import tool

@tool
def hello(your_name: str) -> str:
    """
    This tool returns a greeting message.

    Args:
        your_name: The name to greet
    """
    word = "Hello World, " + "Hello " + your_name 
    return word

In [ ]:
from smolagents import CodeAgent
agent = CodeAgent(tools=[hello], model=model)
agent.run(
    "My name is maris, use the hello tool to greet me, and tell me what the greeting message is?"
)

## Tool Definition Guidelines

- The description should clearly explain what the tool does, providing the LLM with sufficient context. The tool's return value should describe what the task produces. For example, a model_download_tool.
- Input and output types must be specified.
- The description must include an "Args:" section describing each parameter (with type annotations derived from the function signature). The tool description provides the LLM with the explanation it needs to use the tool correctly. A good description is essential for proper tool invocation!

In [ ]:
from smolagents import ActionStep

system_prompt_step = agent.memory.system_prompt
#print("The system prompt given to the agent was:")
#print(system_prompt_step.system_prompt)

task_step = agent.memory.steps[0]
print("\n\nThe first task step was:")
print(task_step.task)

for step in agent.memory.steps:
    if isinstance(step, ActionStep):
        if step.error is not None:
            print(f"\nStep {step.step_number} got this error:\n{step.error}\n")
        else:
            print(f"\nStep {step.step_number} got these observations:\n{step.observations}\n")

In [ ]:
import sys
sys.path.append('..')
import airsim
import math
import numpy as np
import cv2
import base64
import os
from openai import OpenAI
# from gdino import GroundingDINOAPIWrapper, visualize
from dds_cloudapi_sdk.tasks.v2_task import create_task_with_local_image_auto_resize
from dds_cloudapi_sdk import Config
from dds_cloudapi_sdk import Client
from dds_cloudapi_sdk.visualization_util import visualize_result
from PIL import Image
import uuid
from smolagents import tool


@tool
def takeoff() -> str:
    """
    Take off the drone. Returns a string indicating whether the action was successful.
    """
    client = airsim.MultirotorClient()#run in some machine of airsim,otherwise,set ip="" of airsim
    client.confirmConnection()
    client.enableApiControl(True)
    client.armDisarm(True)
    client.takeoffAsync().join()

    return "success"



@tool
def land() -> str:
    """
    Land the drone. Returns a string indicating whether the action was successful.
    """
    client = airsim.MultirotorClient()#run in some machine of airsim,otherwise,set ip="" of airsim
    client.landAsync().join()

    return "success"

In [ ]:
agent = CodeAgent(tools=[takeoff, land], model=model)
agent.run(
    "Take off the drone, wait 3 seconds, then land it."
)